# Lab 2 — Detection and tracking of ships and vehicles: use the tools first
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trongan93/dl-space-2026/blob/main/notebooks/Week4_Lab2_Detection_Tracking.ipynb)  ·  repo: [trongan93/dl-space-2026](https://github.com/trongan93/dl-space-2026)

**Deep Learning in Space Technology Applications · 115-1 · Week 4 (Mon 5 Oct 2026, TA-led workshop) · due Sunday 11 Oct 2026, 23:59 on i-School Plus**

| | |
|---|---|
| Name / student ID (or team) | *fill in* |
| Runtime | Google Colab, **GPU runtime** (Runtime → Change runtime type → T4). CPU works but is ~10× slower. |

### What you will do tonight
1. Run a **pretrained oriented-box detector** (YOLO26-OBB, trained on the DOTA aerial dataset) on aerial/satellite images: planes, ships, vehicles, harbours, storage tanks.
2. Read its outputs as **tensors**: boxes, classes, confidences — the output tensor of Week 3's contract, now real.
3. **Evaluate honestly**: IoU between predicted and labelled boxes, precision/recall at a confidence threshold, mAP on a held-out set. Sweep the threshold and watch precision and recall trade.
4. Break it: **GSD** (downsample the image) and **object size** — connect to Week 2.
5. **Track**: run a tracker (ByteTrack) over a frame sequence, count objects with persistent IDs, and see the difference between detecting in each frame and tracking across frames.
6. Fill the answer cells. The code is given; the marks are for the answers and one small experiment of your own.

Next week (10/12) the instructor explains what is inside the network you are about to use. Tonight you learn what it *does*, how to *measure* it, and where it *fails*.

## 0 · Setup (about two minutes)

In [ ]:
import importlib, subprocess, sys
try: import ultralytics
except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
import os, glob, json, math, time, numpy as np, matplotlib.pyplot as plt, cv2, torch
from ultralytics import YOLO
from ultralytics.utils.downloads import safe_download
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())
DEVICE = 0 if torch.cuda.is_available() else "cpu"

## 1 · Data: DOTA, a public aerial benchmark
[DOTA](https://captain-whu.github.io/DOTA/) is 2 800 aerial/satellite images (Google Earth, GF-2, JL-1) with 188 000 **oriented bounding boxes** in 15 classes — ship, plane, small/large vehicle, harbor, storage tank … GSD ranges from ~0.1 m to ~1 m, so most targets are 10–200 pixels across.

We use two Ultralytics-hosted subsets that download automatically: `dota8` (8 images, 1 MB, to test the pipeline) and `dota128`. The **full DOTAv1 validation set** (used in Part 4, ~2 GB) is optional — run it if the room's network allows.

In [ ]:
from ultralytics.data.utils import check_det_dataset
ds8 = check_det_dataset("dota8.yaml")          # downloads to ./datasets/dota8 on first call
NAMES = ds8["names"]; print(len(NAMES), "classes:", list(NAMES.values()))
imgs8 = sorted(glob.glob(str(ds8["path"] / "images" / "*" / "*.jpg")) + glob.glob(str(ds8["path"] / "images" / "*" / "*.png")))
print(len(imgs8), "images, e.g.", os.path.basename(imgs8[0]))

## 2 · Run the pretrained detector
`yolo26n-obb.pt` (2.4 M parameters, ~6 MB, released Jan 2026) was trained on DOTAv1. If the download fails the cell falls back to `yolo11n-obb.pt` (2.7 M). Run it on one image and read the result as tensors.

In [ ]:
WEIGHTS = "yolo26n-obb.pt"                            # 2026 model; yolo11n-obb.pt is the fallback
try:
    model = YOLO(WEIGHTS)                             # downloads the weights on first call
except Exception as e:
    print("yolo26n-obb.pt not available (", repr(e)[:60], ") -> using yolo11n-obb.pt"); WEIGHTS = "yolo11n-obb.pt"; model = YOLO(WEIGHTS)
print("weights:", WEIGHTS, "| parameters: %.2f M" % (sum(p.numel() for p in model.model.parameters()) / 1e6))
img_path = imgs8[0]
t0 = time.time(); res = model.predict(img_path, imgsz=1024, conf=0.25, device=DEVICE, verbose=False)[0]; dt = time.time() - t0
print(f"inference {dt*1000:.0f} ms on {DEVICE}")
obb = res.obb
print("boxes  (N, 5) [cx, cy, w, h, angle]:", obb.xywhr.shape)
print("classes (N,):", obb.cls.int().tolist())
print("conf    (N,):", [round(c, 2) for c in obb.conf.tolist()])
print("class names:", [NAMES[int(c)] for c in obb.cls.tolist()])
plt.figure(figsize=(10, 10)); plt.imshow(res.plot()[:, :, ::-1]); plt.axis("off"); plt.title(os.path.basename(img_path)); plt.show()

**Read the shapes.** The detector returns `(N, 5)` oriented boxes plus `(N,)` classes and `(N,)` confidences — exactly the *object detection* row of the Week 3 task-tensor table, with one extra number (angle) because ships and vehicles are long and rotated. Every downstream number tonight is computed from these three arrays.

## 3 · Run on all eight images and look at the class mix

In [ ]:
results = model.predict(imgs8, imgsz=1024, conf=0.25, device=DEVICE, verbose=False)
import collections
counts = collections.Counter(NAMES[int(c)] for r in results for c in r.obb.cls.tolist())
print("detections per class:", dict(counts))
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, r in zip(axes.ravel(), results):
    ax.imshow(r.plot(line_width=2)[:, :, ::-1]); ax.set_title(f"{os.path.basename(r.path)}  ({len(r.obb)} boxes)", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 4 · Evaluate honestly — IoU, precision, recall, mAP
Detection metrics work per **object**, not per pixel: a predicted box counts as a true positive if it matches a labelled box of the same class with IoU ≥ 0.5, and each label can be matched once. `model.val()` does this and reports **mAP50** and **mAP50-95** per class.

`dota8` is tiny (it is a smoke test, and the model has seen these images during training). For a real number set `FULL = True` to download the DOTAv1 validation split (~2 GB, 458 images) — the images the model was *not* trained on.

In [ ]:
FULL = False
metrics = model.val(data="DOTAv1.yaml" if FULL else "dota8.yaml", imgsz=1024, batch=8, device=DEVICE, plots=True, verbose=False)
print("mAP50 = %.3f   mAP50-95 = %.3f" % (metrics.box.map50, metrics.box.map))
per_class = {NAMES[int(i)]: round(float(m), 3) for i, m in zip(metrics.box.ap_class_index, metrics.box.ap50)}
print("AP50 per class:", per_class)

### 4b · Sweep the confidence threshold
The detector outputs a confidence per box. The threshold you choose trades precision against recall — the Week 3 mission-cost slide, now measured. We do it by hand on the images, so you see the counting.

In [ ]:
from ultralytics.utils.metrics import batch_probiou
def load_labels(img_path, names_n=15):
    # DOTA/YOLO-OBB label: class x1 y1 x2 y2 x3 y3 x4 y4 (normalised)
    lp = img_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    if not os.path.exists(lp): return np.zeros((0, 9))
    return np.loadtxt(lp, ndmin=2)
def poly_to_xywhr(poly_norm, w, h):
    pts = (poly_norm.reshape(-1, 4, 2) * np.array([w, h])).astype(np.float32)
    out = []
    for p in pts:
        (cx, cy), (bw, bh), ang = cv2.minAreaRect(p); out.append([cx, cy, bw, bh, math.radians(ang)])
    return torch.tensor(out, dtype=torch.float32)

def pr_at(conf_thr, iou_thr=0.5):
    tp = fp = fn = 0
    for r in model.predict(imgs8, imgsz=1024, conf=0.001, device=DEVICE, verbose=False):
        h, w = r.orig_shape; lab = load_labels(r.path)
        keep = r.obb.conf >= conf_thr
        pred = r.obb.xywhr[keep].cpu(); pcls = r.obb.cls[keep].cpu().int()
        if len(lab) == 0: fp += len(pred); continue
        gt = poly_to_xywhr(lab[:, 1:], w, h); gcls = torch.tensor(lab[:, 0]).int()
        if len(pred) == 0: fn += len(gt); continue
        iou = batch_probiou(pred, gt).numpy() * (pcls[:, None] == gcls[None, :]).numpy()
        matched_gt = set()
        for i in np.argsort(-r.obb.conf[keep].cpu().numpy()):
            j = int(np.argmax(iou[i]));
            if iou[i, j] >= iou_thr and j not in matched_gt: tp += 1; matched_gt.add(j)
            else: fp += 1
        fn += len(gt) - len(matched_gt)
    return tp / max(1, tp + fp), tp / max(1, tp + fn), tp, fp, fn
ths = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
rows = [(t, *pr_at(t)) for t in ths]
print(" conf  precision  recall   TP  FP  FN")
for t, p, r_, tp, fp, fn in rows: print(f" {t:4.2f}   {p:6.3f}    {r_:6.3f}  {tp:3d} {fp:3d} {fn:3d}")
plt.figure(figsize=(5, 4)); plt.plot([x[1] for x in rows], [x[2] for x in rows], "o-", color="#3066BE")
for t, p, r_, *_ in rows: plt.annotate(f"{t:.2f}", (p, r_), fontsize=8, xytext=(3, 3), textcoords="offset points")
plt.xlabel("precision"); plt.ylabel("recall"); plt.title("precision-recall as the confidence threshold moves"); plt.grid(alpha=.3); plt.show()

## 5 · Break it — GSD and object size
Downsample an image by 2×, 4×, 8× (GSD gets coarser: 0.5 m → 1 m → 2 m → 4 m if the original was 0.5 m) and count what the detector still finds. This is the Week 2 "pixels on target" slide, measured.

In [ ]:
img = cv2.imread(imgs8[0]); H, W = img.shape[:2]
fig, axes = plt.subplots(1, 4, figsize=(22, 6)); summary = []
for ax, f in zip(axes, [1, 2, 4, 8]):
    small = cv2.resize(img, (W // f, H // f), interpolation=cv2.INTER_AREA)
    r = model.predict(small, imgsz=max(256, 1024 // f), conf=0.25, device=DEVICE, verbose=False)[0]
    c = collections.Counter(NAMES[int(k)] for k in r.obb.cls.tolist()); summary.append((f, len(r.obb), dict(c)))
    ax.imshow(r.plot(line_width=1)[:, :, ::-1]); ax.axis("off"); ax.set_title(f"{f}x coarser GSD: {len(r.obb)} detections")
plt.tight_layout(); plt.show()
for f, n, c in summary: print(f"{f}x coarser: {n:3d} detections  {c}")

## 6 · Tracking — detections become objects with identity
A detector answers *where are the ships in this frame*. A tracker answers *is this the same ship as in the last frame*: it links detections across frames by predicting motion and matching boxes (ByteTrack keeps low-confidence boxes alive between frames, which is why it tolerates the occasional missed detection).

Real satellite video (Jilin-1, SkySat) is not freely redistributable, so this cell **simulates** a 60-frame sequence: it cuts ship crops out of a DOTA harbour image and moves them along straight paths over the sea. The physics is fake; the tracking problem is real.

In [ ]:
# --- build a simulated satellite-video clip from detected ships ---
harbour = None
for r in results:
    if (r.obb.cls == 1).sum() >= 3: harbour = r; break
if harbour is None: harbour = results[0]
img = cv2.imread(harbour.path); H, W = img.shape[:2]
ships = harbour.obb.xyxy[harbour.obb.cls == 1].cpu().numpy().astype(int)[:6]
crops = [img[max(0,y1):y2, max(0,x1):x2].copy() for x1, y1, x2, y2 in ships if (y2-y1) > 8 and (x2-x1) > 8]
# sea background: median colour of the darkest 20% of pixels, plus gentle noise
gray = img.mean(2); sea_col = np.median(img[gray < np.percentile(gray, 20)], axis=0)
rng = np.random.default_rng(0); FH, FW, N = 512, 768, 60
paths = [(rng.uniform(40, FW-140), rng.uniform(40, FH-80), rng.uniform(-3, 3), rng.uniform(-2, 2)) for _ in crops]
os.makedirs("simvideo", exist_ok=True)
for t in range(N):
    frame = (np.ones((FH, FW, 3)) * sea_col + rng.normal(0, 6, (FH, FW, 3))).clip(0, 255).astype(np.uint8)
    for (x0, y0, vx, vy), crop in zip(paths, crops):
        x, y = int(x0 + vx*t), int(y0 + vy*t); ch, cw = crop.shape[:2]
        if 0 <= x < FW-cw and 0 <= y < FH-ch: frame[y:y+ch, x:x+cw] = crop
    if 20 <= t < 26: frame[:, FW//3:FW//3+90] = (200, 200, 200)     # a cloud bar crosses the scene: occlusion
    cv2.imwrite(f"simvideo/{t:03d}.jpg", frame)
out = cv2.VideoWriter("simvideo.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 10, (FW, FH))
for t in range(N): out.write(cv2.imread(f"simvideo/{t:03d}.jpg"))
out.release(); print(len(crops), "ships moving over", N, "frames -> simvideo.mp4 (frames 20-25 partly occluded)")

In [ ]:
# --- detect per frame vs track across frames ---
det_counts, track_ids, per_frame_ids = [], set(), []
for t in range(N):
    frame = cv2.imread(f"simvideo/{t:03d}.jpg")
    r = model.track(frame, imgsz=768, conf=0.2, classes=[1], tracker="bytetrack.yaml", persist=True, device=DEVICE, verbose=False)[0]
    det_counts.append(len(r.obb))
    ids = r.obb.id.int().tolist() if r.obb.id is not None else []
    per_frame_ids.append(ids); track_ids.update(ids)
    if t in (5, 22, 50):
        plt.figure(figsize=(9, 6)); plt.imshow(r.plot(line_width=2)[:, :, ::-1]); plt.axis("off"); plt.title(f"frame {t}: {len(r.obb)} detections, ids {ids}"); plt.show()
print("ships placed:", len(crops))
print("detections per frame (min / median / max):", min(det_counts), int(np.median(det_counts)), max(det_counts))
print("distinct track IDs over the clip:", sorted(track_ids), "->", len(track_ids), "tracks")
plt.figure(figsize=(10, 3)); plt.plot(det_counts, label="detections in frame", color="#B05A2B")
plt.plot([len(set(i)) for i in per_frame_ids], label="active track IDs", color="#3066BE"); plt.axvspan(20, 26, color="#DDDDDD", label="occlusion")
plt.xlabel("frame"); plt.legend(frameon=False); plt.title("per-frame detection vs tracking through an occlusion"); plt.show()

**What to look for.** During the occlusion (frames 20–25) the detector loses one or two ships; the tracker should keep their IDs alive for a few frames and re-attach them afterwards. If the number of distinct IDs is larger than the number of ships placed, the tracker *fragmented* a track (a ship got a new identity) — the classic tracking failure. Try `botsort.yaml` and compare.

## 7 · Your experiment (choose one, 15 minutes)
- **A. Class confusion.** Run the detector on an image with both large and small vehicles; make a confusion table between predicted and labelled classes (use `load_labels`).
- **B. Tiling.** Take one large DOTA image, run at `imgsz=640` on the whole image vs on 640-px tiles with 20 % overlap; compare the number of small vehicles found.
- **C. Tracker settings.** Change the ships' speed (`vx, vy` range) or the occlusion length and find where ByteTrack starts to fragment tracks.
- **D. Your seed problem.** If your team's problem is detection, run the detector on one of your own images (upload it) and write down what it finds and misses.

In [ ]:
# your experiment here

## 8 · ✏️ YOUR ANSWERS (marked)
1. **Output tensor.** Write the shapes of the three arrays the detector returns for one image and say what each column of the box array means. Why does a ship detector want an *oriented* box?
2. **Threshold.** From your precision–recall table: which confidence threshold would you choose for (a) counting all ships in a harbour for a statistics report, (b) alerting a coast-guard patrol boat? Justify with FP/FN costs.
3. **GSD.** At which downsampling factor did small vehicles disappear, and ships? Convert the factors to metres per pixel assuming 0.5 m for the original, and relate to the "5–10 pixels across" rule.
4. **Tracking.** How many ships did you place, how many distinct IDs did the tracker produce, and what happened during the occlusion? What would go wrong with real satellite video that this simulation hides (name two things)?
5. **Your experiment.** One paragraph: what you changed, what you measured, what you concluded.

✏️ **YOUR ANSWER**

1. 

2. 

3. 

4. 

5. 

---
### Checklist before you export
- [ ] GPU runtime used; all cells executed
- [ ] mAP line and the threshold table printed
- [ ] GSD figure and tracking figure visible
- [ ] Five answers written; experiment cell has code and output

*DOTA: Xia et al., "DOTA: A Large-scale Dataset for Object Detection in Aerial Images", CVPR 2018. Detector: Ultralytics YOLO26-OBB (AGPL-3.0; YOLO11-OBB fallback). Tracker: ByteTrack (Zhang et al., ECCV 2022).*